In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, roc_auc_score
)
from sklearn.utils import resample

# Matplotlib defaults
plt.rcParams["figure.dpi"] = 120
sns.set(style="whitegrid", rc={"axes.spines.right": False, "axes.spines.top": False})

train = pd.read_csv("satellite_train.csv")
test  = pd.read_csv("satellite_test.csv")

X_train = train[[f"V{i}" for i in range(1, 37)]].values   # V1–V36
y_train = train["V37"].values                              # target
X_test  = test [[f"V{i}" for i in range(1, 37)]].values
y_test  = test ["V37"].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

def print_title(s):
    print("\n" + "="*len(s))
    print(s)
    print("="*len(s))

# ==================================================
# Q1 
# ==================================================
print_title("Q1 — Feature Correlation")

corr = pd.DataFrame(X_train, columns=[f"V{i}" for i in range(1, 37)]).corr()
mean_abs_corr = corr.abs().values[np.triu_indices_from(corr, 1)].mean()
print(f"Average absolute inter-feature correlation: {mean_abs_corr:.3f}")

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": .7})
plt.title("Feature Correlation Matrix (V1–V36)")
plt.tight_layout()
plt.show()

# ==================================================
# Q2 
# ==================================================
print_title("Q2 — LDA assumption check on selected pairs")

# Choose pairs: good (V20 vs V29), bad (V15 vs V19)
pairs = [("V20","V29"), ("V15","V19")]
pair_idxs = [(20,29), (15,19)]  # 1-based indexes for readability

def plot_lda_pair(vx, vy):
    ix = int(vx[1:]) - 1
    iy = int(vy[1:]) - 1

    Xtr = X_train[:, [ix, iy]]
    Xte = X_test [:, [ix, iy]]

    lda2 = LinearDiscriminantAnalysis()
    lda2.fit(Xtr, y_train)

    # Grid for boundary
    x_min, x_max = Xtr[:,0].min()-1, Xtr[:,0].max()+1
    y_min, y_max = Xtr[:,1].min()-1, Xtr[:,1].max()+1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = lda2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    # Scatter + boundary
    plt.figure(figsize=(6,5))
    plt.contourf(xx, yy, Z, alpha=0.15, levels=np.unique(y_train), cmap="tab10")
    sc = plt.scatter(Xtr[:,0], Xtr[:,1], c=y_train, cmap="tab10", s=8, alpha=0.8)
    plt.xlabel(vx); plt.ylabel(vy)
    plt.title(f"LDA boundary on {vx} vs {vy}")
    plt.tight_layout()
    plt.show()

for vx, vy in pairs:
    plot_lda_pair(vx, vy)

# ==================================================
# Q3 
# ==================================================
print_title("Q3 — Thresholds & ROC (Logistic vs LDA)")

def threshold_metrics(y_true, y_prob, n=200):
    thresholds = np.linspace(0, 1, n)
    rows = []
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        acc = accuracy_score(y_true, pred)
        prec = precision_score(y_true, pred, zero_division=0)
        rec = recall_score(y_true, pred)
        f1 = f1_score(y_true, pred)
        tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        j = rec + spec - 1
        rows.append((t, acc, prec, rec, f1, spec, j))
    df = pd.DataFrame(rows, columns=["threshold","accuracy","precision","recall","f1","specificity","youden"])
    return df

# Binary subproblem: class 1 vs rest
y_train_bin = (y_train == 1).astype(int)
y_test_bin  = (y_test  == 1).astype(int)

log = LogisticRegression(max_iter=1000)
lda = LinearDiscriminantAnalysis()

log.fit(X_train, y_train_bin)
lda.fit(X_train, y_train_bin)

yprob_log = log.predict_proba(X_test)[:,1]
yprob_lda = lda.predict_proba(X_test)[:,1]

res_log = threshold_metrics(y_test_bin, yprob_log)
res_lda = threshold_metrics(y_test_bin, yprob_lda)

opt_log_y = res_log.loc[res_log["youden"].idxmax()]
opt_log_f = res_log.loc[res_log["f1"].idxmax()]
opt_lda_y = res_lda.loc[res_lda["youden"].idxmax()]
opt_lda_f = res_lda.loc[res_lda["f1"].idxmax()]

print("Logistic optimal thresholds:",
      f"Youden={opt_log_y.threshold:.3f}, F1-opt={opt_log_f.threshold:.3f}")
print("LDA optimal thresholds:",
      f"Youden={opt_lda_y.threshold:.3f}, F1-opt={opt_lda_f.threshold:.3f}")

# Plot Metrics vs Threshold — Logistic
plt.figure(figsize=(9,5))
for m in ["accuracy","precision","recall","f1","specificity"]:
    plt.plot(res_log["threshold"], res_log[m], label=m.capitalize())
plt.axvline(opt_log_y.threshold, color="red", linestyle="--", label="Youden")
plt.axvline(opt_log_f.threshold, color="purple", linestyle="--", label="F1-opt")
plt.xlabel("Probability threshold"); plt.ylabel("Score")
plt.title("Metrics vs Threshold (Logistic Regression)")
plt.legend(ncol=3)
plt.tight_layout()
plt.show()

# Plot Metrics vs Threshold — LDA
plt.figure(figsize=(9,5))
for m in ["accuracy","precision","recall","f1","specificity"]:
    plt.plot(res_lda["threshold"], res_lda[m], label=m.capitalize())
plt.axvline(opt_lda_y.threshold, color="red", linestyle="--", label="Youden")
plt.axvline(opt_lda_f.threshold, color="purple", linestyle="--", label="F1-opt")
plt.xlabel("Probability threshold"); plt.ylabel("Score")
plt.title("Metrics vs Threshold (LDA)")
plt.legend(ncol=3)
plt.tight_layout()
plt.show()

# ROC with AUC and marked points
fpr_log, tpr_log, _ = roc_curve(y_test_bin, yprob_log)
fpr_lda, tpr_lda, _ = roc_curve(y_test_bin, yprob_lda)
auc_log = auc(fpr_log, tpr_log)
auc_lda = auc(fpr_lda, tpr_lda)

plt.figure(figsize=(6,6))
plt.plot(fpr_log, tpr_log, label=f"Logistic (AUC={auc_log:.3f})")
plt.plot(fpr_lda, tpr_lda, label=f"LDA (AUC={auc_lda:.3f})")
plt.plot([0,1], [0,1], "k--", label="Random")
# mark optimal points
plt.scatter(1-opt_log_y.specificity, opt_log_y.recall, color="red", label="Logistic Youden")
plt.scatter(1-opt_log_f.specificity, opt_log_f.recall, color="purple", label="Logistic F1-opt")
plt.scatter(1-opt_lda_y.specificity, opt_lda_y.recall, color="orange", label="LDA Youden")
plt.scatter(1-opt_lda_f.specificity, opt_lda_f.recall, color="brown", label="LDA F1-opt")
plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC — Logistic vs LDA (Class 1 vs Rest)")
plt.legend()
plt.tight_layout()
plt.show()

# ==================================================
# Q4
# ==================================================
print_title("Q4 — Feature Reduction (Accuracy)")

# Build reduced sets (use scaled data for stability)
df_scaled = pd.DataFrame(X_train_scaled, columns=[f"V{i}" for i in range(1, 37)])
corr_abs = df_scaled.corr().abs()
upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] > 0.9)]

Xtr_corr = df_scaled.drop(columns=to_drop).values
Xte_corr = pd.DataFrame(X_test_scaled, columns=df_scaled.columns).drop(columns=to_drop).values

pca = PCA(n_components=10, random_state=42)
Xtr_pca = pca.fit_transform(X_train_scaled); Xte_pca = pca.transform(X_test_scaled)

sel = SelectKBest(f_classif, k=15)
Xtr_f = sel.fit_transform(X_train_scaled, y_train); Xte_f = sel.transform(X_test_scaled)

strategies = {
    "All features": (X_train_scaled, X_test_scaled),
    "Corr-filtered": (Xtr_corr, Xte_corr),
    "PCA (10 comps)": (Xtr_pca, Xte_pca),
    "ANOVA top-15": (Xtr_f, Xte_f),
}

models = {
    "Logistic": LogisticRegression(max_iter=1000),
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis()
}

rows = []
for sname, (Xtr, Xte) in strategies.items():
    for mname, model in models.items():
        model.fit(Xtr, y_train)
        pred = model.predict(Xte)
        rows.append((sname, mname, accuracy_score(y_test, pred)))

acc_df = pd.DataFrame(rows, columns=["Strategy","Model","Accuracy"])
pivot_acc = acc_df.pivot(index="Strategy", columns="Model", values="Accuracy")
print(pivot_acc.round(4).to_string())

plt.figure(figsize=(9,5))
sns.barplot(data=acc_df, x="Strategy", y="Accuracy", hue="Model", palette="Set2")
plt.title("Accuracy across Feature Reduction Strategies")
plt.ylim(0.65, 0.9)
plt.tight_layout()
plt.show()

# ==================================================
# Q5
# ==================================================
print_title("Q5 — QDA vs Quadratic Logistic")

# Baseline Logistic on scaled data
log_base = LogisticRegression(max_iter=2000).fit(X_train_scaled, y_train)
ypl_lr = log_base.predict_proba(X_test_scaled)
pred_lr = log_base.predict(X_test_scaled)

# Quadratic logistic (quadratic terms + pairwise interactions)
poly = PolynomialFeatures(degree=2, include_bias=False)
Xtr_q = poly.fit_transform(X_train_scaled)
Xte_q = poly.transform(X_test_scaled)

log_quad = LogisticRegression(max_iter=3000).fit(Xtr_q, y_train)
ypl_quad = log_quad.predict_proba(Xte_q)
pred_quad = log_quad.predict(Xte_q)

# QDA
qda = QuadraticDiscriminantAnalysis().fit(X_train_scaled, y_train)
ypl_qda = qda.predict_proba(X_test_scaled)
pred_qda = qda.predict(X_test_scaled)

def macro_table(name, y_true, y_pred, y_proba):
    return dict(
        Model=name,
        Accuracy=accuracy_score(y_true, y_pred),
        MacroF1=f1_score(y_true, y_pred, average="macro"),
        AUC=roc_auc_score(pd.get_dummies(y_true), y_proba, average="macro")
    )

res_q5 = pd.DataFrame([
    macro_table("Logistic", y_test, pred_lr, ypl_lr),
    macro_table("Quadratic Logistic", y_test, pred_quad, ypl_quad),
    macro_table("QDA", y_test, pred_qda, ypl_qda),
])
print(res_q5.round(4))

# Bar chart
melt_q5 = res_q5.melt(id_vars="Model", value_vars=["Accuracy","MacroF1","AUC"],
                      var_name="Metric", value_name="Score")
plt.figure(figsize=(8,5))
sns.barplot(data=melt_q5, x="Metric", y="Score", hue="Model", palette="Set2")
plt.title("Logistic vs Quadratic Logistic vs QDA")
plt.ylim(0.7, 1.0); plt.tight_layout(); plt.show()

# ROC curves (macro via flattening)
plt.figure(figsize=(6,6))
for name, proba in [("Logistic", ypl_lr), ("Quadratic Logistic", ypl_quad), ("QDA", ypl_qda)]:
    fpr, tpr, _ = roc_curve(pd.get_dummies(y_test).values.ravel(), proba.ravel())
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc(fpr,tpr):.3f})")
plt.plot([0,1], [0,1], "k--", label="Random")
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC — QDA vs Quadratic Logistic")
plt.legend(); plt.tight_layout(); plt.show()

# ==================================================
# Q6
# ==================================================
print_title("Q6 — Small-sample regime learning curves")

fractions = [0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
scores = { "LDA": {"acc":[], "f1":[]},
           "Logistic": {"acc":[], "f1":[]},
           "QDA": {"acc":[], "f1":[]} }

for frac in fractions:
    X_sub, y_sub = resample(X_train_scaled, y_train,
                            n_samples=int(len(X_train_scaled)*frac), random_state=42)
    # LDA
    m = LinearDiscriminantAnalysis().fit(X_sub, y_sub)
    p = m.predict(X_test_scaled)
    scores["LDA"]["acc"].append(accuracy_score(y_test, p))
    scores["LDA"]["f1"].append(f1_score(y_test, p, average="macro"))
    # Logistic
    m = LogisticRegression(max_iter=2000).fit(X_sub, y_sub)
    p = m.predict(X_test_scaled)
    scores["Logistic"]["acc"].append(accuracy_score(y_test, p))
    scores["Logistic"]["f1"].append(f1_score(y_test, p, average="macro"))
    # QDA
    m = QuadraticDiscriminantAnalysis().fit(X_sub, y_sub)
    p = m.predict(X_test_scaled)
    scores["QDA"]["acc"].append(accuracy_score(y_test, p))
    scores["QDA"]["f1"].append(f1_score(y_test, p, average="macro"))

# Accuracy curve
plt.figure(figsize=(8,5))
for name in scores:
    plt.plot(fractions, scores[name]["acc"], "o-", label=f"{name} Accuracy")
plt.xlabel("Fraction of Training Data"); plt.ylabel("Accuracy")
plt.title("Learning Curves — Accuracy")
plt.legend(); plt.grid(True, alpha=.3); plt.tight_layout(); plt.show()

# Macro-F1 curve
plt.figure(figsize=(8,5))
for name in scores:
    plt.plot(fractions, scores[name]["f1"], "o-", label=f"{name} Macro-F1")
plt.xlabel("Fraction of Training Data"); plt.ylabel("Macro-F1")
plt.title("Learning Curves — Macro-F1")
plt.legend(); plt.grid(True, alpha=.3); plt.tight_layout(); plt.show()

# ==================================================
# Q7 
# ==================================================
print_title("Q7 — Robustness to sensor quirks (gain shift)")

def macro_f1_after_fit(model, Xtr, Xte):
    model.fit(Xtr, y_train)
    return f1_score(y_test, model.predict(Xte), average="macro")

# Baseline (clean)
baseline_f1 = {
    "Logistic": macro_f1_after_fit(LogisticRegression(max_iter=2000), X_train_scaled, X_test_scaled),
    "LDA":      macro_f1_after_fit(LinearDiscriminantAnalysis(),        X_train_scaled, X_test_scaled),
    "QDA":      macro_f1_after_fit(QuadraticDiscriminantAnalysis(),     X_train_scaled, X_test_scaled),
}

# Train-only gain shift on V5 (+10, clipped) then scale with same scaler (as in spec)
X_shift = X_train.copy()
X_shift[:, 4] = np.clip(X_shift[:, 4] + 10, 0, 255)  # V5 → index 4
X_shift_scaled = scaler.transform(X_shift)

gainshift_f1 = {
    "Logistic": macro_f1_after_fit(LogisticRegression(max_iter=2000), X_shift_scaled, X_test_scaled),
    "LDA":      macro_f1_after_fit(LinearDiscriminantAnalysis(),        X_shift_scaled, X_test_scaled),
    "QDA":      macro_f1_after_fit(QuadraticDiscriminantAnalysis(),     X_shift_scaled, X_test_scaled),
}

df_q7 = pd.DataFrame({
    "Model": ["Logistic","LDA","QDA"],
    "Baseline_F1": [baseline_f1[m] for m in ["Logistic","LDA","QDA"]],
    "GainShift_F1": [gainshift_f1[m] for m in ["Logistic","LDA","QDA"]],
})
df_q7["Delta"] = df_q7["GainShift_F1"] - df_q7["Baseline_F1"]
print(df_q7.round(4))

# Bar plot
melt_q7 = df_q7.melt(id_vars="Model", value_vars=["Baseline_F1","GainShift_F1"],
                     var_name="Condition", value_name="Macro-F1")
plt.figure(figsize=(7,5))
sns.barplot(data=melt_q7, x="Model", y="Macro-F1", hue="Condition", palette="Set2")
plt.title("Robustness under Train-only Gain Shift (+10 on V5)")
plt.ylim(max(0.0, melt_q7["Macro-F1"].min()-0.05), min(1.0, melt_q7["Macro-F1"].max()+0.05))
plt.tight_layout(); plt.show()